<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/03.%20Video%20Sync.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Joint Angle Trial**
> MARC - Multidisciplinary AI Research Centre | UOP


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
# @title Store Initialization
from google.colab import drive
import pandas as pd
import os
!pip install -q ultralytics opencv-python pandas scipy
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

import plotly.io as pio
pio.renderers.default = "colab"


In [ ]:
# @title Dataset Summary
store.summary()
store.list_entries(entry_type="experiments")

# Visual Sync Pipeline

### Functions

In [ ]:
# @title Utilities
import cv2
import json
import os
import subprocess
import threading
import numpy as np
import pandas as pd
from scipy import signal
import plotly.graph_objects as go
from ultralytics import YOLO

# YOLO (COCO) 17 keypoint names
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# Core biomechanical keypoints (excluding facial keypoints to reduce noise)
MOTION_KEYPOINTS = [
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# Initialize YOLO model once with a lock for thread-safe GPU inference
yolo_model = YOLO('yolo11x-pose.pt')
yolo_lock = threading.Lock()
store_lock = threading.Lock()

def analyze_keypoints(video_path, batch_size=32, min_conf=0.25):
    """
    Thread-safe batched inference on the GPU using Ultralytics generator stream.
    Filters out (0, 0) unpredicted points and low-confidence detections as NaN.
    """
    all_frames_data = []

    with yolo_lock:
        results_generator = yolo_model.predict(
            source=video_path,
            batch=batch_size,
            stream=True,
            verbose=False,
            device=0,
            half=True  # Enables FP16 for speedup on Tensor Cores
        )

        for frame_idx, results in enumerate(results_generator):
            frame_data = {"frame": frame_idx}

            if results.keypoints is not None and len(results.keypoints.xy) > 0:
                confs = results.keypoints.conf.cpu().numpy()
                # Select the person with the highest mean keypoint confidence if multiple detected
                best_person_idx = int(np.argmax(np.mean(confs, axis=1)))

                kpts = results.keypoints.xy[best_person_idx].cpu().numpy()
                scores = confs[best_person_idx]

                if len(kpts) == 17:
                    for i, name in enumerate(KEYPOINT_NAMES):
                        x = float(kpts[i][0])
                        y = float(kpts[i][1])
                        s = float(scores[i])
                        # Ultralytics sets undetected keypoints to (0.0, 0.0) with low confidence
                        if s < min_conf or (x == 0.0 and y == 0.0):
                            frame_data[f"{name}_x"] = np.nan
                            frame_data[f"{name}_y"] = np.nan
                            frame_data[f"{name}_score"] = s
                        else:
                            frame_data[f"{name}_x"] = x
                            frame_data[f"{name}_y"] = y
                            frame_data[f"{name}_score"] = s
                else:
                    for name in KEYPOINT_NAMES:
                        frame_data[f"{name}_x"] = np.nan
                        frame_data[f"{name}_y"] = np.nan
                        frame_data[f"{name}_score"] = 0.0
            else:
                for name in KEYPOINT_NAMES:
                    frame_data[f"{name}_x"] = np.nan
                    frame_data[f"{name}_y"] = np.nan
                    frame_data[f"{name}_score"] = 0.0

            all_frames_data.append(frame_data)

    df = pd.DataFrame(all_frames_data)
    output_json_path = video_path.replace('.mp4', '_keypoints.json')
    df.to_json(output_json_path, orient='records')
    print(f"Keypoints saved to {output_json_path}")
    return df


def calculate_sync_lag(front_df, side_df, plot=False):
    """
    Calculates the frame lag between the front and side videos.
    Lag = t_front - t_side:
      - Positive lag (> 0): Event happens at a later frame index in front_df (front video started earlier;
        side camera started recording later). Trim first `best_lag` frames of front video.
      - Negative lag (< 0): Event happens at a later frame index in side_df (side video started earlier;
        front camera started recording later). Trim first `|best_lag|` frames of side video.
    """
    def get_movement_signal(df):
        x_cols = [f"{name}_x" for name in MOTION_KEYPOINTS]
        y_cols = [f"{name}_y" for name in MOTION_KEYPOINTS]
        s_cols = [f"{name}_score" for name in MOTION_KEYPOINTS]

        kx = df[x_cols].copy()
        ky = df[y_cols].copy()
        ks = df[s_cols].fillna(0.0).to_numpy()

        # Mask out low-confidence detections per joint before interpolation
        low_conf_mask = ks < 0.3
        kx[low_conf_mask] = np.nan
        ky[low_conf_mask] = np.nan

        # Interpolate each joint independently to avoid (0,0) spikes or full-frame dropouts
        kx = kx.interpolate(method='linear', limit_direction='both').ffill().bfill().fillna(0.0).to_numpy()
        ky = ky.interpolate(method='linear', limit_direction='both').ffill().bfill().fillna(0.0).to_numpy()

        dx = np.diff(kx, axis=0)
        dy = np.diff(ky, axis=0)

        # Emphasize vertical (Y) velocity, which is shared across both orthographic views,
        # and gate frame-to-frame differences by joint confidence
        pair_conf = np.minimum(ks[:-1], ks[1:])
        joint_speed = np.sqrt(0.25 * dx**2 + dy**2) * (pair_conf >= 0.3)

        raw_sig = np.sum(joint_speed, axis=1)
        if len(raw_sig) >= 5:
            raw_sig = signal.medfilt(raw_sig, kernel_size=5)
        return raw_sig

    sig_front = get_movement_signal(front_df)
    sig_side  = get_movement_signal(side_df)

    sig_front_norm = (sig_front - np.mean(sig_front)) / (np.std(sig_front) + 1e-8)
    sig_side_norm  = (sig_side - np.mean(sig_side)) / (np.std(sig_side) + 1e-8)

    correlation = signal.correlate(sig_front_norm, sig_side_norm, mode='full')
    lags = signal.correlation_lags(len(sig_front_norm), len(sig_side_norm), mode='full')

    best_lag = int(lags[np.argmax(correlation)])

    if plot:
        # Original Signals
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(y=sig_front_norm, mode='lines', name='Front Movement', opacity=0.8))
        fig1.add_trace(go.Scatter(y=sig_side_norm, mode='lines', name='Side Movement', opacity=0.8))
        fig1.update_layout(title="Original Un-Synced Movement Signals", template="plotly_dark", height=400)
        fig1.show()

        # Synced Signals
        fig2 = go.Figure()
        if best_lag > 0:
            x1 = np.arange(len(sig_front_norm))
            x2 = np.arange(len(sig_side_norm)) + best_lag
        else:
            x1 = np.arange(len(sig_front_norm)) - best_lag
            x2 = np.arange(len(sig_side_norm))

        fig2.add_trace(go.Scatter(x=x1, y=sig_front_norm, mode='lines', name='Front Movement (Synced)', opacity=0.8))
        fig2.add_trace(go.Scatter(x=x2, y=sig_side_norm, mode='lines', name='Side Movement (Synced)', opacity=0.8))
        fig2.update_layout(title=f"Aligned Movement Signals (Side Lag: {best_lag} frames)", template="plotly_dark", height=400)
        fig2.show()

    return best_lag


# Pre-parsed skeleton lookup array to avoid Python dictionary/tuple overhead inside the frame loop
SKELETON_INDICES = np.array([
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
    (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
    (13, 15), (12, 14), (14, 16)
], dtype=np.int32)

def generate_annotated_video(video_path, keypoints_df):
    """
    High-speed vectorized skeleton drawing with fast encoding and NaN checks.
    """
    CONFIDENCE_THRESHOLD = 0.4
    cap = cv2.VideoCapture(video_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0

    out_path = video_path.replace('.mp4', '_annotated.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    total_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), len(keypoints_df))
    x_cols = [f"{name}_x" for name in KEYPOINT_NAMES]
    y_cols = [f"{name}_y" for name in KEYPOINT_NAMES]
    s_cols = [f"{name}_score" for name in KEYPOINT_NAMES]

    all_kpts_x = keypoints_df[x_cols].to_numpy()
    all_kpts_y = keypoints_df[y_cols].to_numpy()
    all_scores = keypoints_df[s_cols].to_numpy()

    frame_idx = 0
    while cap.isOpened() and frame_idx < total_frames:
        ret, frame = cap.read()
        if not ret:
            break

        scores = all_scores[frame_idx]
        kx = all_kpts_x[frame_idx]
        ky = all_kpts_y[frame_idx]
        valid_mask = (scores >= CONFIDENCE_THRESHOLD) & (~np.isnan(kx)) & (~np.isnan(ky))

        # Draw skeleton limbs
        for p1, p2 in SKELETON_INDICES:
            if valid_mask[p1] and valid_mask[p2]:
                pt1 = (int(kx[p1]), int(ky[p1]))
                pt2 = (int(kx[p2]), int(ky[p2]))
                cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_4)

        # Draw keypoint dots
        for i in np.where(valid_mask)[0]:
            cv2.circle(frame, (int(kx[i]), int(ky[i])), 4, (0, 0, 255), -1)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved to {out_path}")
    return out_path


def generate_synced_video(front_video_path, side_video_path, side_lag_frames):
    """
    Frame-accurate side-by-side video synchronization using multi-threaded FFmpeg.
    Uses the `trim=start_frame` filter and `setpts` to avoid keyframe-seeking drift.
    """
    out_path = os.path.join(os.path.dirname(front_video_path), 'synced_side_by_side.mp4')
    if os.path.exists(out_path):
        os.remove(out_path)

    cap = cv2.VideoCapture(front_video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()

    if side_lag_frames > 0:
        # Front video started earlier -> trim leading frames from front video
        trim_f = f"trim=start_frame={int(side_lag_frames)},setpts=PTS-STARTPTS,"
        trim_s = "setpts=PTS-STARTPTS,"
    elif side_lag_frames < 0:
        # Side video started earlier -> trim leading frames from side video
        trim_f = "setpts=PTS-STARTPTS,"
        trim_s = f"trim=start_frame={int(abs(side_lag_frames))},setpts=PTS-STARTPTS,"
    else:
        trim_f = "setpts=PTS-STARTPTS,"
        trim_s = "setpts=PTS-STARTPTS,"

    filter_complex = (
        f"[0:v]{trim_f}fps={fps:.4f},scale=-2:720[v0];"
        f"[1:v]{trim_s}fps={fps:.4f},scale=-2:720[v1];"
        f"[v0][v1]hstack=inputs=2:shortest=1[v]"
    )

    cmd = (
        f"ffmpeg -y -threads 4 "
        f"-i \"{front_video_path}\" "
        f"-i \"{side_video_path}\" "
        f"-filter_complex \"{filter_complex}\" "
        f"-map \"[v]\" -c:v libx264 -preset ultrafast -crf 23 -an \"{out_path}\""
    )

    result = subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if result.returncode == 0 and os.path.exists(out_path):
        print(f"Synced video saved to {out_path}")
        return out_path
    else:
        print("FFmpeg synchronization fallback: check video input paths.")
        return None

In [ ]:
# @title Visual Estimation Pipeline
import os
from concurrent.futures import ThreadPoolExecutor

def visual_estimation_pipeline(entry_id, store, generate_annotated=True, generate_synced=True, plot=True):
    """
    Pipeline for visual estimation of a specific experiment.

    Args:
        entry_id (str): The ID of the experiment to process.
        store (OpenDatasetStore): The initialized store object.
        generate_annotated (bool): Whether to generate skeleton-annotated videos.
        generate_synced (bool): Whether to generate a side-by-side synchronized video.
        plot (bool): Whether to render interactive Plotly sync charts.

    Returns:
        tuple: (front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path)
    """
    entry_type = "experiments"

    # Create experiment folder
    exp_folder = f"/content/{entry_id}"
    os.makedirs(exp_folder, exist_ok=True)
    print(f"Setting up directory: {exp_folder}")

    # Download videos directly to variable paths
    print("Downloading front_video...")
    front_video_path = store.download_related_file(
        entry_type=entry_type,
        entry_id=entry_id,
        label="front_video",
        dest_folder=exp_folder
    )

    print("Downloading side_video...")
    side_video_path = store.download_related_file(
        entry_type=entry_type,
        entry_id=entry_id,
        label="side_video",
        dest_folder=exp_folder
    )

    if not front_video_path or not os.path.exists(front_video_path) or not side_video_path or not os.path.exists(side_video_path):
        print("Error: Could not locate both downloaded video files.")
        return None, None, None, None, None, None

    print("\n--- Analyzing Keypoints ---")
    front_df = analyze_keypoints(front_video_path, batch_size=32)
    side_df  = analyze_keypoints(side_video_path, batch_size=32)

    print("\n--- Calculating Sync Lag ---")
    side_lag = calculate_sync_lag(front_df, side_df, plot=plot)
    print(f"Calculated side lag: {side_lag} frames")

    front_annotated = None
    side_annotated = None
    synced_video = None

    if generate_annotated:
        print("\n--- Generating Annotated Videos in Parallel ---")
        with ThreadPoolExecutor(max_workers=2) as render_pool:
            f_front = render_pool.submit(generate_annotated_video, front_video_path, front_df)
            f_side  = render_pool.submit(generate_annotated_video, side_video_path, side_df)
            front_annotated = f_front.result()
            side_annotated  = f_side.result()

    if generate_synced:
        print("\n--- Generating Synchronized Video (FFmpeg Ultrafast) ---")
        synced_video = generate_synced_video(front_video_path, side_video_path, side_lag)

    print("\nPipeline complete!")
    return front_df, side_df, side_lag, front_annotated, side_annotated, synced_video

In [ ]:
# @title Save Results
def save_visual_estimation_results(entry_id, store, front_df, side_df, side_lag,
                                   front_annotated_path=None, side_annotated_path=None,
                                   synced_video_path=None):
    """
    Thread-safe saving of results from the visual estimation pipeline back to the OpenDatasetStore.
    """
    entry_type = "experiments"
    with store_lock:
        print(f"--- Saving results for {entry_id} to OpenDatasetStore ---")

        # 1. Save Metadata (Lag Frames)
        if side_lag is not None:
            store.edit_entry(
                entry_type=entry_type,
                entry_id=entry_id,
                **{"video_sync_side_lag_frames": int(side_lag)}
            )
            print(f"✓ Saved metadata: video_sync_side_lag_frames = {int(side_lag)}")

        # 2. Save Keypoint DataFrames
        if front_df is not None and not front_df.empty:
            store.add_processed_data(
                entry_type=entry_type,
                entry_id=entry_id,
                tag="front_video_keypoints",
                df=front_df
            )
            print("✓ Saved front_video_keypoints dataframe")

        if side_df is not None and not side_df.empty:
            store.add_processed_data(
                entry_type=entry_type,
                entry_id=entry_id,
                tag="side_video_keypoints",
                df=side_df
            )
            print("✓ Saved side_video_keypoints dataframe")

        # 3. Save Generated Videos
        def save_video(file_path, label):
            if file_path and os.path.exists(file_path):
                try:
                    store.add_related_file(
                        entry_type=entry_type,
                        entry_id=entry_id,
                        label=label,
                        local_path=file_path
                    )
                    print(f"✓ Saved video file for label: {label}")
                except AttributeError:
                    print(f"⚠ Could not automatically save {label}. Store API might require a different method.")
                except Exception as e:
                    print(f"⚠ Error saving {label}: {e}")

        save_video(synced_video_path, "synced_video")
        save_video(front_annotated_path, "front_annotated_video")
        save_video(side_annotated_path, "side_annotated_video")

        print("--- Save Complete! ---")


### Run

In [ ]:
# @title Run Full Pipeline

target_entry_id = "experiments_0001" #@param {type:"string"}

print(f"Starting end-to-end process for {target_entry_id}...")

# 1. Run the pipeline to generate DataFrames and videos
front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path = visual_estimation_pipeline(
    entry_id=target_entry_id,
    store=store,
    generate_annotated=True,
    generate_synced=True,
    plot=True
)


In [ ]:
# @title Ensure data is valid, then save
if front_df is not None and side_df is not None:
    # Save all results back to the OpenDatasetStore
    save_visual_estimation_results(
        entry_id=target_entry_id,
        store=store,
        front_df=front_df,
        side_df=side_df,
        side_lag=side_lag,
        front_annotated_path=front_annotated_path,
        side_annotated_path=side_annotated_path,
        synced_video_path=synced_video_path
    )
    print(f"\nSuccessfully completed processing and saving for {target_entry_id}!")
else:
    print(f"\nFailed to process {target_entry_id}. Please check the logs above for errors.")

In [ ]:
# @title Deleted Saved Files Experiment
store.delete_related_file(entry_type='experiments', entry_id=target_entry_id, label='synced_video')
store.delete_related_file(entry_type='experiments', entry_id=target_entry_id, label='front_annotated_video')
store.delete_related_file(entry_type='experiments', entry_id=target_entry_id, label='side_annotated_video')


In [ ]:
# @title Parallel Run: Visual Estimation & Auto-Save
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

experiments_to_run = [
    "experiments_0023",
    "experiments_0024",
    "experiments_0025",
    "experiments_0026",
    "experiments_0027",
    "experiments_0028",
    "experiments_0029",
    "experiments_0030",
    "experiments_0031",
    "experiments_0032",
    "experiments_0033",
    "experiments_0034",
    "experiments_0035",
    "experiments_0036",
    "experiments_0037",
]

def run_single_experiment(entry_id):
    """Worker to process and save a single experiment."""
    print(f"\n[START] Pipeline started for: {entry_id}")
    t0 = time.time()

    # 1. Run the visual estimation pipeline
    results = visual_estimation_pipeline(
        entry_id=entry_id,
        store=store,
        generate_annotated=True,
        generate_synced=True,
        plot=True
    )

    if not results or results[0] is None or results[1] is None:
        print(f"\n[FAIL] Processing failed for: {entry_id}")
        return entry_id, False

    front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path = results

    # 2. Save back using thread-safe save function
    save_visual_estimation_results(
        entry_id=entry_id,
        store=store,
        front_df=front_df,
        side_df=side_df,
        side_lag=side_lag,
        front_annotated_path=front_annotated_path,
        side_annotated_path=side_annotated_path,
        synced_video_path=synced_video_path
    )

    dt = time.time() - t0
    print(f"\n[DONE] Finished & saved {entry_id} in {dt:.1f}s")
    return entry_id, True

# -------------------------------------------------------------------------
# Concurrency configuration
# max_workers=3 balances OpenCV decoding, thread-locked CUDA inference,
# and disk write operations without context thrashing.
# -------------------------------------------------------------------------
MAX_WORKERS = 3

print(f"Launching batch execution with {MAX_WORKERS} parallel workers...")
start_all = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(run_single_experiment, exp_id): exp_id for exp_id in experiments_to_run}

    for future in as_completed(futures):
        exp_id = futures[future]
        try:
            _, success = future.result()
        except Exception as e:
            print(f"[ERROR] Exception during processing of {exp_id}: {e}")

print(f"\n{'='*50}\nAll experiments completed in {time.time() - start_all:.1f}s\n{'='*50}")